In [41]:
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn import global_mean_pool


class GAT_FeatureExtractor(torch.nn.Module):
    def __init__(self, input_layer=320, hidden_channels=16, output_layer=240):
        super(GAT_FeatureExtractor,self).__init__()
        self.conv1 = GATConv(input_layer, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, output_layer)

    def forward(self, x, edge_index, node_weights):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = global_mean_pool(x, batch)
        return x

In [42]:
class GAT_WeightedMean(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = GATConv(320, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index, node_weights):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        
        # Normalize node_weights per graph
        norm_weights = node_weights / node_weights.sum()
        x = x * norm_weights.unsqueeze(-1)
        return x.sum(dim=0)


In [43]:
from torch_geometric.nn import GlobalAttention

class GAT_AttentionPool(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = GATConv(320, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, hidden_channels)
        self.attn_pool = GlobalAttention(gate_nn=torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1)
        ))

    def forward(self, x, edge_index, node_weights):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)  # all nodes in same graph
        return self.attn_pool(x, batch)


In [44]:
from torch_geometric.nn import GCNConv

class GCN_WeightedSum(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(320, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index, node_weights):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        return (x * node_weights.unsqueeze(-1)).sum(dim=0)


In [45]:
from torch_geometric.nn import TransformerConv

class Transformer_WeightedMean(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = TransformerConv(320, hidden_channels, heads=1)
        self.conv2 = TransformerConv(hidden_channels, hidden_channels, heads=1)
        self.conv3 = TransformerConv(hidden_channels, hidden_channels, heads=1)

    def forward(self, x, edge_index, node_weights, edge_attr):
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr).relu()
        x = self.conv3(x, edge_index, edge_attr)

        norm_weights = node_weights / node_weights.sum()
        x = x * norm_weights.unsqueeze(-1)
        return x.sum(dim=0)


In [46]:
from torch_geometric.data import Dataset
import torch
import os

class PatientGraphDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()
        self.root_dir = root_dir
        self.files = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.endswith('.pt')]
        self.transform = transform

    def len(self):
        return len(self.files)

    def get(self, idx):
        self.graph, self.df = torch.load(self.files[idx], weights_only=False)
        if self.transform:
            self.graph = self.transform(self.graph)
        return self.graph


In [47]:
from torch_geometric.loader import DataLoader
import torch

downsample_size = 10000
full_dataset = PatientGraphDataset(f"/dsi/efroni-lab/sbm/chen2/graph_data/{downsample_size}/0.05_True_True/")
# full_dataset = DataLoader(full_dataset, batch_size=90, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [49]:
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from collections import defaultdict


# Evaluate models
models = {
    "GAT_FeatureExtractor": GAT_FeatureExtractor(input_layer=320, hidden_channels=16, output_layer=240),
    "GAT_WeightedMean": GAT_WeightedMean(16),
    "GAT_AttentionPool": GAT_AttentionPool(16),
    "GCN_WeightedSum": GCN_WeightedSum(16),
    # "Transformer_WeightedMean": Transformer_WeightedMean(16)
}

results = {}
detailed_results = defaultdict(dict)

for name, model in models.items():
    model = model.to(device)
    model.eval()
    patient_vectors = []
    labels = []

    with torch.no_grad():
        for data in tqdm(full_dataset, desc=f"Evaluating {name}"):
            data = data.to(device)
            if name == "Transformer_WeightedMean":
                vec = model(data.x, data.edge_index, data.node_weights, data.edge_attr)
            else:
                vec = model(data.x, data.edge_index, data.node_weights)
            patient_vectors.append(vec.cpu().numpy())
            labels.append(data.y.item())

    X = np.vstack(patient_vectors)
    y = np.array(labels)
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
    clf = SVC(kernel='rbf')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy

results

/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  warnings.warn(out)
Evaluating GAT_FeatureExtractor:   0%|          | 0/91 [00:00<?, ?it/s]

Evaluating GCN_WeightedSum: 100%|██████████| 91/91 [00:02<00:00, 30.86it/s]


{'GAT_FeatureExtractor': 0.5217391304347826,
 'GAT_WeightedMean': 0.5217391304347826,
 'GAT_AttentionPool': 0.5217391304347826,
 'GCN_WeightedSum': 0.5652173913043478}